# SwitchLingua — download our labels and choose a version

Every token of the SwitchLingua corpus (234,172 conversations, 15 language pairs) already has a language label. This notebook downloads those labels from our **private** Hugging Face dataset and builds the prediction targets in the form you choose:

| Label | Meaning |
|---|---|
| `lang` | `L1` (first language), `L2` (second language), or `-` (punctuation, digits: skipped) |
| `y_sw` | `1` if the **next** token switches language, `0` if not, `-1` where undefined |
| `y_len` | at a switch, how many tokens the new run lasts (integer; 0 elsewhere) |
| `y_dur` | the class of `y_len`: `small` (1–2), `medium` (3–6), `large` (7+); empty elsewhere |
| `censored` | the run is still open when the text ends, so `y_len` is a lower bound |
| `dur_known` | the duration class is certain: the run is closed, or it is cut off but already 7+ |

Which model produced the underlying word labels, and how accurate each is, is in `docs/labelling_models.md` in the GitHub repo.

## How to run

1. **Access.** Your Hugging Face account needs access to `cristae/switchlingua-tagged-final` (ask Cristae) and, for sub-word labels only, to the gated [`Shelton1013/SwitchLingua_text`](https://huggingface.co/datasets/Shelton1013/SwitchLingua_text).
2. **Token.** On Kaggle, add a secret named `HF_TOKEN` (Add-ons → Secrets) holding a read token. Locally, put `HF_TOKEN=...` in `.env`.
3. **Settings.** Turn Internet on. A CPU is enough.
4. **Choose** in the Config cell below, then *Run All*. The output files appear in `/kaggle/working` (Output tab).

Word labels take about a minute; sub-word labels about 5 (they tokenize the original text with XLM-R).

## Config

In [ ]:
# ---- Choose your labels -----------------------------------------------------
LEVEL = "word"           # "word":    one label per word (Chinese/Japanese/Korean: per character)
                         # "subword": one label per XLM-R piece (what an XLM-R model reads)
NAMES = "script_words"   # how names count:
                         #   "script_words" (default) a name takes the language its context implies
                         #   "transparent"            names are skipped, like punctuation
                         #   "script"                 every Latin-script name is English
PAIRS = None             # None = all 15, or e.g. [("Spanish", "English"), ("German", "French")]
SPLITS = None            # None = all, or e.g. ["train"]   (row_id % 10: 0-7 train, 8 val, 9 test)
LIMIT = None             # None = every row, or e.g. 200 rows per pair for a quick look
EXPORT = "parquet"       # "parquet", "csv" or "both"

LABELS_REPO = "cristae/switchlingua-tagged-final"

## Setup

In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "regex", "wordfreq"], check=True)

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("labels_out")
os.makedirs(os.path.join(WORK, "src"), exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, WORK)

TOKEN = os.environ.get("HF_TOKEN")
if not TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if not TOKEN:
    try:
        from dotenv import dotenv_values
        for env in (".env", "../.env"):
            TOKEN = TOKEN or dotenv_values(env).get("HF_TOKEN")
    except Exception:
        pass
assert LEVEL in ("word", "subword"), LEVEL
assert NAMES in ("script_words", "transparent", "script"), NAMES
assert EXPORT in ("parquet", "csv", "both"), EXPORT
print("working dir:", WORK, "| HF token:", "found" if TOKEN else "MISSING - add the HF_TOKEN secret")

## Label code
The live `src/` label modules, embedded verbatim.

In [ ]:
%%writefile src/__init__.py
# package marker

In [ ]:
%%writefile src/routing.py
"""Language-pair -> tagging-strategy lookup.

Two regimes, confirmed empirically against the cached SwitchLingua CSVs:

* ``unicode`` - first_language uses a script disjoint from Latin, so a token's
  script *is* its language. Exact, no model. ~68.7% of rows.
* ``model``   - first_language is Latin-script (or romanized), so only
  vocabulary separates it from the second language. Needs constrained LID.

Note the plan's routing table listed Arabic's pair as Arabic<->French; the data
says ``Arabic_eng.csv`` is Arabic<->English, and French appears as a *second*
language only in ``German_French.csv``. Nothing here hardcodes the second
language - callers pass the row's own ``second_language``.
"""

SCRIPT_DISJOINT_CONFIRMED = {
    "Cantonese", "Chinese", "Japanese", "Korean", "Russian", "Arabic",
}

# Resolved from the plan's `unicode_pending_check` gate: 54.7% of Hindi rows are
# romanized Hinglish ("Woh kal Prime Minister Carney se milne wali hain"), so
# Devanagari-vs-Latin is not a reliable split for this language. Routed to model.
NEEDS_MODEL = {
    "German", "Philippines", "Vietnamese", "Malay", "Spanish", "French",
    "Italian", "Hindi",
}

# fastText lid.176 label codes.
LANG_TO_CODE = {
    "German": "de", "Philippines": "tl", "Vietnamese": "vi", "Malay": "ms",
    "Spanish": "es", "French": "fr", "Italian": "it", "English": "en",
    "Hindi": "hi", "Russian": "ru", "Arabic": "ar", "Chinese": "zh",
    "Cantonese": "zh", "Japanese": "ja", "Korean": "ko",
}

# Script expected for each unicode-routed first_language. A token in one of
# these scripts is the first language; a Latin token is the second language.
_EXPECTED_SCRIPT = {
    "Cantonese": {"Han"},
    "Chinese": {"Han"},
    "Japanese": {"Han", "Hiragana", "Katakana"},
    "Korean": {"Hangul"},
    "Russian": {"Cyrillic"},
    "Arabic": {"Arabic"},
    "Hindi": {"Devanagari"},
}

# fastText's `hi` model is trained on Devanagari. Romanized Hindi tokens are
# out of distribution for every label, so rows here are flagged low-confidence
# rather than silently trusted.
ROMANIZATION_RISK = {"Hindi"}


# --- tier 3: context-aware LinCE BERT token classifiers ---------------------
#
# Measured on hand-labelled tokens from the real rows (NE excluded), fastText
# vs. the spaeng model used cross-lingually:
#
#     language      fastText   spaeng    -> routed to
#     German           100%      80%        fasttext
#     French            90%      68%        fasttext
#     Vietnamese        90%      88%        fasttext   (tie; diacritics carry it)
#     Italian           96%     100%        spaeng
#     Spanish           94%     100%        spaeng
#     Philippines       86%      94%        spaeng
#     Malay             75%      88%        spaeng
#     Hindi          broken       -         hineng
#
# spaeng transfers as a generic Latin-L1-vs-English discriminator to Romance and
# Austronesian languages but *regresses* Germanic/French, so routing is per
# language and empirical, not by family. hineng does not transfer at all (17%
# Malay / 64% Tagalog) - it is Hindi-only.
#
# Sample sizes are small (n=26-48 per language); treat Vietnamese and Italian,
# where the gap is inside a couple of tokens, as ties rather than firm results.
MODEL_REPO = {
    "spaeng": "sagorsarker/codeswitch-spaeng-lid-lince",
    "hineng": "sagorsarker/codeswitch-hineng-lid-lince",
}
MODEL_L1_LABEL = {"spaeng": "spa", "hineng": "hin"}
MODEL_FOR = {
    "Hindi": "hineng",
    "Spanish": "spaeng", "Italian": "spaeng",
    "Malay": "spaeng", "Philippines": "spaeng",
}


# --- tier 4: dictionary frequency comparison (measured best) ----------------
#
# Measured on data/gold_eval.json (324 hand-labelled tokens) and confirmed on
# 767 rows against the dataset's own cs_ratio_result:
#
#   language      dict-freq   best model      ratio err (dict vs prod)
#   Malay            100%      88% mBERT        0.103 vs 0.183
#   Philippines      100%      94% mBERT        0.147 vs 0.222
#   French           100%      90% fastText     0.121 vs 0.153
#   German           100%     100% fastText     0.133 vs 0.151
#   Vietnamese       100%      90% fastText     0.084 vs 0.093
#   Spanish          100%     100% mBERT        0.105 vs 0.104
#   Italian          100%     100% mBERT        0.132 vs 0.124
#   Hindi (roman)     66%     100% mBERT        - model required
#
# dict-freq wins or ties on every Latin-script pair and needs no model at all.
# It also handles a non-English second language (the German<->French rows),
# which neither LinCE model can, so fastText and mBERT-spaeng both retire.
#
# Romanized Hindi is the sole exception: wordfreq's `hi` list is Devanagari and
# romanized Hinglish has no standard orthography, so no word list can cover it.
# Four independent techniques agree the context-aware models are required there
# (fastText 56%, dict-freq 66%, mBERT 100%, XLM-R 100%, CANINE 98%).
USES_DICT = {
    "Spanish", "Italian", "Malay", "Philippines",
    "German", "French", "Vietnamese",
}




# --- tier 5: AnE-LID (current production for every "<Latin L1> vs English") --
#
# The gold set above saturated (every tier at ~99-100% on 26-59 self-labelled
# tokens), so the decision was made on human-annotated public data instead
# (scripts/eval_public.py, L1/EN tokens only):
#
#   set                      n     dict-freq   mBERT    AnE-LID
#   LinCE spaeng dev      31,667     94.1%     98.8%     99.4%
#   LinCE hineng dev      12,303     80.5%     97.4%     98.8%
#   Denglisch (de-en)     59,485     96.2%     53.0%     99.9%  (likely seen in training)
#
# dict-freq's "100%" on the gold set was ~94-96% on human labels. AnE-LID is
# multi-pair trained, so it also covers romanized Hindi and replaces mBERT
# hineng there. It only answers "English or not", so German->French keeps
# dict-freq. Where AnE and dict-freq agree, tokens are 99.3-100% correct.
ANE_L2 = "English"


def ane_to_language(label: str, first_language: str, second_language: str = "English") -> str:
    """Map a raw AnE-LID label to the row's language. ``Mixed`` (an English stem
    in L1 morphology, "rewatchen") goes to L1, its grammatical host."""
    if label == "English":
        return second_language
    if label in ("notEnglish", "Mixed"):
        return first_language
    return "other"


def route(first_language: str, second_language: str = "English") -> str:
    """Return ``"unicode"``, ``"ane"`` or ``"dict"`` for a row.

    Hindi routes per *row* rather than per language - see
    ``dispatch._hindi_is_devanagari``. Devanagari rows are script-disjoint from
    English and go to the exact tagger; only romanized rows need the model.
    ``"bert"`` (mBERT hineng) is retired but still dispatchable for comparison.
    """
    if first_language in SCRIPT_DISJOINT_CONFIRMED:
        return "unicode"
    if second_language != ANE_L2:
        return "dict"
    return "ane"


def lang_code(language: str) -> str | None:
    """Map a dataset language name to its fastText lid.176 label code."""
    return LANG_TO_CODE.get(language)


def expected_script(language: str) -> set[str]:
    """Unicode script name(s) that indicate ``language`` in a mixed token stream."""
    return _EXPECTED_SCRIPT.get(language, {"Latin"})


# --- V1 quality tiers (measured, not assumed) -------------------------------
#
# The plan's escalation rule needs a per-pair accuracy number. Manual labelling
# was not required to get one: the six script-disjoint languages are tagged
# *exactly*, so their disagreement with the dataset's own `computed_ratio`
# (-0.134 .. +0.117, mean |diff| ~0.075) measures the noise in that reference
# signal rather than any tagger error. That band is the yardstick - a
# model-routed pair sitting inside it is behaving like an exact tagger; one far
# outside it is genuinely broken.
#
# Measured on a 150-row-per-language stratified sample (see notebook cell):
#
#   PASS  (inside the exact-tagger band)   Vietnamese .056  Spanish .086
#                                          German .118  French .128  Italian .129
#   FAIL  (2-5x outside it)                Malay .277  Hindi .369  Philippines .406
#
# This inverts the plan's expectation, which predicted German/Malay/Tagalog
# would pass easily and the Romance pairs would fail on shared function words.
# The Romance pairs are fine; lid.176 simply has far less Malay/Tagalog training
# data, and romanized Hindi is not in its label space at all.
V1_PASSES = {"Vietnamese", "Spanish", "German", "French", "Italian"}
V1_FAILS_NEEDS_V2 = {"Malay", "Philippines", "Hindi"}


# Measured token accuracy after routing each language to its best tagger.
# `low` now means "still under the plan's 90% bar with the best option tried".
# Accuracy of the tagger each language is now routed to, on the gold set.
# Every Latin pair scored 100% under dict-freq - but on only 26-50 tokens each,
# self-labelled. `provisional` marks that: good enough to ship, not yet a
# number to publish. The review tool exists to replace these.
_MEASURED_ACCURACY = {
    "Spanish": 1.00, "Italian": 1.00, "Malay": 1.00, "Philippines": 1.00,
    "German": 1.00, "French": 1.00, "Vietnamese": 1.00,   # dict-freq
    "Hindi": 1.00,                                        # mBERT hineng
}


def tagger_quality(first_language: str, second_language: str = "English") -> str:
    """``exact`` | ``provisional`` - how far to trust a row's tags.

    ``exact`` is correct by construction (script-disjoint). ``provisional``
    means measured at >=90% on a small self-labelled sample; human review is
    what promotes it to a defensible number.
    """
    if first_language in SCRIPT_DISJOINT_CONFIRMED:
        return "exact"
    return "provisional" if _MEASURED_ACCURACY.get(first_language, 0) >= 0.90 else "low"

In [ ]:
%%writefile src/tokenizer.py
"""Mixed-granularity tokenizer.

CJK does not tokenize on whitespace - a run of Han/Kana/Hangul between two
English words may hold several words with no space - so those scripts are cut
per character. Everything else is cut per word.

Two fixes over the pattern in the plan:

1. ``[A-Za-z]+`` splits accented Latin: "dificil" became ``dif|i|cil``, which
   shatters every Spanish/French/German/Vietnamese word carrying a diacritic
   and would badly inflate run counts. ``\\p{Script=Latin}`` covers the accented
   range, since U+00E9 and friends *are* Latin script.
2. Cyrillic, Arabic and Devanagari are space-delimited, so word-level is right
   for them; the plan's fallback branch would have cut them per character.
"""
import regex as re

# Order matters: character-level CJK alternatives are tried before the
# word-level ones, and the single-character fallback is last.
_PATTERN = re.compile(
    r"\p{Script=Han}"
    r"|\p{Script=Hiragana}"
    r"|\p{Script=Katakana}"
    r"|\p{Script=Hangul}"
    r"|[\p{Script=Latin}\p{M}]+(?:['’][\p{Script=Latin}\p{M}]+)*"
    r"|[\p{Script=Cyrillic}\p{M}]+"
    r"|[\p{Script=Arabic}\p{M}]+"
    r"|[\p{Script=Devanagari}\p{M}]+"
    r"|\d+"
    r"|[^\s]"
)


def tokenize_mixed(text: str) -> list[str]:
    """Split text into CJK-character tokens and word tokens for other scripts."""
    return _PATTERN.findall(text)

In [ ]:
%%writefile src/dict_tagger.py
"""Dictionary-lookup tagging - the obvious approach, measured.

Two variants of "look the word up in each language's dictionary":

``tag_naive``      if the word exists in the second language's list, call it the
                   second language. This is the intuitive rule, and it fails on
                   lexical overlap: most *common* words exist in both lists.
``tag_frequency``  compare how common the word is in each language and take the
                   higher. Strictly better, and the reason is instructive - the
                   question is never "is this word in the dictionary" but "which
                   language is it more characteristic of".

Frequencies come from `wordfreq` (Zipf scale: ~7 = very common, 0 = absent).
"""
from __future__ import annotations

from functools import lru_cache

from wordfreq import zipf_frequency

LANG_TO_CODE = {
    "English": "en", "Spanish": "es", "German": "de", "French": "fr",
    "Italian": "it", "Malay": "ms", "Philippines": "fil",
    "Vietnamese": "vi", "Hindi": "hi",
}


@lru_cache(maxsize=500_000)
def zipf(word: str, code: str) -> float:
    return zipf_frequency(word, code)


def _is_word(token: str) -> bool:
    return any(ch.isalpha() for ch in token)


def tag_naive(tokens: list[str], first_language: str,
              second_language: str) -> list[tuple[str, str]]:
    """"If it's in the second language's dictionary, it's the second language.\""""
    l2 = LANG_TO_CODE.get(second_language, "en")
    out = []
    for t in tokens:
        if not _is_word(t):
            out.append((t, "other"))
        elif zipf(t.lower(), l2) > 0:
            out.append((t, second_language))
        else:
            out.append((t, first_language))
    return out


def tag_frequency(tokens: list[str], first_language: str, second_language: str,
                  margin: float = 0.0) -> list[tuple[str, str]]:
    """Whichever language the word is more characteristic of wins."""
    l1 = LANG_TO_CODE.get(first_language)
    l2 = LANG_TO_CODE.get(second_language, "en")
    out = []
    for t in tokens:
        if not _is_word(t) or l1 is None:
            out.append((t, "other" if not _is_word(t) else first_language))
            continue
        w = t.lower()
        f1, f2 = zipf(w, l1), zipf(w, l2)
        if f1 == 0 and f2 == 0:
            out.append((t, "other"))          # unknown to both: proper noun etc.
        else:
            out.append((t, second_language if f2 > f1 + margin else first_language))
    return out


def overlap_stats(tokens: list[str], first_language: str,
                  second_language: str) -> dict:
    """How often is a word present in BOTH dictionaries? That is the ceiling
    on the naive rule: every such word is a coin flip for it."""
    l1 = LANG_TO_CODE.get(first_language)
    l2 = LANG_TO_CODE.get(second_language, "en")
    words = [t.lower() for t in tokens if _is_word(t)]
    if not words or l1 is None:
        return dict(words=0, both=0, neither=0, only_l1=0, only_l2=0)
    both = neither = only1 = only2 = 0
    for w in words:
        a, b = zipf(w, l1) > 0, zipf(w, l2) > 0
        both += a and b
        neither += not a and not b
        only1 += a and not b
        only2 += b and not a
    return dict(words=len(words), both=both, neither=neither,
                only_l1=only1, only_l2=only2)

In [ ]:
%%writefile src/unicode_tagger.py
"""Script-based tagging, for language pairs whose scripts are disjoint.

Exact by construction: if the first language is written in Han and the second
in Latin, a token's script settles which language it belongs to. No model, no
threshold, nothing to validate beyond the script table itself.
"""
import regex as re

from .routing import expected_script

_SCRIPTS = ["Han", "Hiragana", "Katakana", "Hangul", "Devanagari",
            "Cyrillic", "Arabic", "Latin"]
_SCRIPT_RE = {name: re.compile(fr"\p{{Script={name}}}") for name in _SCRIPTS}


def char_script(ch: str) -> str:
    for name in _SCRIPTS:
        if _SCRIPT_RE[name].match(ch):
            return name
    return "other"


def tag_token_unicode(token: str) -> str:
    """Return the token's single script name, or ``"other"`` if mixed/unscripted.

    Combining marks are ignored when deciding, so an accented Latin token stays
    ``Latin`` rather than falling through to ``other``.
    """
    scripts = {char_script(c) for c in token if not c.isspace()}
    scripts.discard("other")
    if len(scripts) == 1:
        return scripts.pop()
    return "other"


def tag_tokens_unicode(tokens: list[str], first_language: str,
                       second_language: str) -> list[tuple[str, str]]:
    """Tag tokens as the row's first or second language by script.

    Returns ``(token, language)`` pairs, where language is the row's own
    ``first_language`` / ``second_language`` value, or ``"other"`` for
    punctuation and digits.
    """
    l1_scripts = expected_script(first_language)
    l2_scripts = expected_script(second_language)
    out = []
    for tok in tokens:
        script = tag_token_unicode(tok)
        if script in l1_scripts:
            out.append((tok, first_language))
        elif script in l2_scripts:
            out.append((tok, second_language))
        else:
            out.append((tok, "other"))
    return out

In [ ]:
%%writefile src/dispatch.py
"""Single entry point that routes a token stream to the right tagger.

Kept separate so ``segments``, ``inspection`` and ``persist`` share one copy of
the branch, and so the heavy imports (torch, the models) stay lazy - a
unicode-routed row loads none of them.
"""
from __future__ import annotations

import regex as re

from .routing import ane_to_language, route

_DEVANAGARI = re.compile(r"\p{Script=Devanagari}")
_LATIN = re.compile(r"\p{Script=Latin}")


def _hindi_is_devanagari(tokens: list[str]) -> bool:
    """Hindi rows come in two scripts; only the romanized half needs a model.

    Routing per *language* sent Devanagari rows to the LinCE model, which is
    trained on romanized Hinglish and so is out of distribution for them - while
    the Unicode tagger handles Devanagari-vs-Latin exactly. Decide per row.
    """
    dev = sum(1 for t in tokens if _DEVANAGARI.search(t))
    lat = sum(1 for t in tokens if _LATIN.search(t))
    return dev > lat


def row_strategy(tokens: list[str], first_language: str, second_language: str) -> str:
    """``route`` plus the per-row Devanagari check for Hindi."""
    strategy = route(first_language, second_language)
    if (strategy in ("ane", "bert") and first_language == "Hindi"
            and _hindi_is_devanagari(tokens)):
        return "unicode"
    return strategy


def _devanagari_is_hindi(tagged, first_language):
    """Romanized-majority Hindi rows still carry some Devanagari words, and the
    LinCE model called 24,113 of them English. Script settles those exactly."""
    if first_language != "Hindi":
        return tagged
    return [(t, first_language if _DEVANAGARI.search(t) else lang) for t, lang in tagged]


def _names_from_context(tagged, is_entity, ane_labels, first_language, second_language):
    """Script-tagged rows: a Latin-script *name* takes AnE-LID's contextual label.

    Script alone calls every Latin token English, which is right for
    "Game of Thrones" but invents a switch for "Messi" in a Cantonese sentence.
    AnE-LID, run on a window around the name, tells the two apart. Names in the
    row's own script (中關村) stay L1, and non-entity tokens keep their script
    label. An empty label means the name was never scored; script stands.
    """
    out = []
    for (t, lang), ent, lab in zip(tagged, is_entity, ane_labels):
        if ent and lab and _LATIN.search(t):
            lang = ane_to_language(lab, first_language, second_language)
        out.append((t, lang))
    return out


def tag_tokens(tokens: list[str], first_language: str, second_language: str,
               ane_labels: list[str] | None = None,
               is_entity: list[bool] | None = None) -> list[tuple[str, str]]:
    """Tag tokens as first/second language via whichever tier the pair needs.

    ``ane_labels`` are precomputed raw AnE-LID word labels (``persist.ane_labels``
    batches them over the corpus). For ``ane`` rows they are the labels; without
    them the model runs on this row. For script-tagged rows they are optional
    and only cover the windows around Latin-script names (see
    ``_names_from_context``); ``""`` marks unscored positions.
    """
    strategy = row_strategy(tokens, first_language, second_language)
    if strategy == "ane":
        from .ane_tagger import tag_tokens_ane
        return _devanagari_is_hindi(
            tag_tokens_ane(tokens, first_language, second_language, labels=ane_labels),
            first_language)
    if strategy == "unicode":
        from .unicode_tagger import tag_tokens_unicode
        tagged = tag_tokens_unicode(tokens, first_language, second_language)
        if (ane_labels is not None and is_entity is not None
                and len(ane_labels) == len(tokens) == len(is_entity)):
            tagged = _names_from_context(tagged, is_entity, ane_labels,
                                         first_language, second_language)
        return tagged
    if strategy == "bert":
        from .bert_tagger import tag_tokens_bert
        return _devanagari_is_hindi(
            tag_tokens_bert(tokens, first_language, second_language), first_language)
    if strategy == "dict":
        from .dict_tagger import tag_frequency
        return tag_frequency(tokens, first_language, second_language)
    from .model_tagger import tag_tokens_model
    return tag_tokens_model(tokens, first_language, second_language)

In [ ]:
%%writefile src/segments.py
"""Run-length encoding of tagged tokens, plus the dataset builders.

The plan leaves one decision open (line 174): whether an ``"other"`` tag
(punctuation, digits) breaks a run or is absorbed into the surrounding
language. Absorbing is cleaner for the downstream goals, so ``absorb_other``
does that - with one guard the naive version gets wrong: an ``other`` token
sitting *between two different languages* must not bridge them into a single
run. "hola, hello" is two runs, not one.

Named entities: every token keeps a *language* label, and entity-ness travels
separately as an ``is_entity`` flag. How names count is a policy, not a label
(``resolve_names``; every edge case is pinned in tests/test_name_strategies.py).
AnE-NER fragments names, so ``repair_spans`` mends them first for every policy.

* ``names="script_words"`` (default) - a name is a switch only with visible
  evidence. A Latin-script name in a non-Latin row is English ("Messi",
  "Severance Season 2" inside Arabic), since the writer chose the Latin script;
  a name in the row's own script stays L1. In Latin-script rows the name's
  words decide when they are clearly one language's ("Severance Season" ->
  English, "Union europeenne" -> French); otherwise it takes its surroundings
  ("Ayer Leo Messi marco" -> Spanish), and may not open a run on its own.
* ``names="transparent"`` - names are no language and are skipped (LinCE's
  convention). Stored as the main ablation.
* ``names="script"`` - every Latin-script name is English. Stored too; adds a
  one-word switch for each Messi / Monza / EU inside a Latin-script sentence.
* ``names="hybrid"`` - titles keep their language, other names take context.
* ``names="language"`` - names count as AnE-LID tagged them.
"""
from __future__ import annotations

import ast

import regex as re

from .routing import route
from .tokenizer import tokenize_mixed

NE = "NE"
NAME_POLICIES = ("hybrid", "language", "transparent", "script", "script_words")
# A name counts as a switch only with visible evidence: a change of script, or
# words that are clearly one language's. See tests/test_name_strategies.py.
DEFAULT_NAMES = "script_words"

# A function word: short and extremely frequent in its language (wordfreq Zipf).
# Measured: of/the/and/de/la/der/dan/ng/cua all score 6.5-7.9; the most common
# content words in candidate names (game, real, united, league) stay <= 5.8.
_FUNCTION_ZIPF = 6.4
_FUNCTION_MAXLEN = 4


def _is_function_word(word: str, language: str) -> bool:
    from .dict_tagger import LANG_TO_CODE, zipf
    code = LANG_TO_CODE.get(language)
    return (code is not None and len(word) <= _FUNCTION_MAXLEN
            and zipf(word.lower(), code) >= _FUNCTION_ZIPF)


_LATIN = re.compile(r"\p{Script=Latin}")
# Languages whose wordfreq list is written in Latin script, so a Latin-script
# name can be looked up in it. Hindi's list is Devanagari: romanized names
# score ~0 there, which would make every Bollywood name "English".
_LATIN_DICT = {"English", "Spanish", "German", "French", "Italian", "Malay",
               "Philippines", "Vietnamese"}
_MARGIN = 1.0      # Zipf gap for a word to count as clearly one language's


def _latin(tok: str) -> bool:
    return bool(_LATIN.search(tok))


def _zipf(word: str, language: str):
    if language not in _LATIN_DICT:
        return None
    from .dict_tagger import LANG_TO_CODE, zipf
    return zipf(word.lower(), LANG_TO_CODE[language])


def _word_side(word: str, first_language: str, second_language: str):
    """Which of the pair a Latin word clearly belongs to, by dictionary margin; None if unclear.

    Names sit in both frequency lists at similar levels (messi 3.8 en / 4.1 es,
    madrid 4.3 / 5.4), so only a >= 1.0 Zipf gap decides: severance, season ->
    English; europeenne, monza -> L1; messi, psg, eu -> unclear.
    """
    z1, z2 = _zipf(word, first_language), _zipf(word, second_language)
    if z1 is None or z2 is None:
        return None
    if z2 - z1 >= _MARGIN:
        return second_language
    if z1 - z2 >= _MARGIN:
        return first_language
    return None


def repair_spans(tokens, is_entity, first_language: str, second_language: str) -> list[bool]:
    """Mend AnE-NER's fragmented Latin-script names before any policy uses them.

    Measured on real output: "Severance Season 2" came back as just
    "Severance", "Game of Thrones" as "Game" + "Thrones", "The University of
    Texas" as "of Texas". Two repairs, Latin script only:

    * bridge a single function word between two name words ("Game [of] Thrones");
    * grow a name over adjacent capitalised words - and numbers to its right -
      unless the word is clearly an ordinary L1 word ("Ayer Leo Messi" keeps
      "Ayer"; German nouns like "Kommission" are never swallowed).
    Non-Latin names are left exactly as tagged.
    """
    mask = list(is_entity)
    n = len(tokens)
    for k in range(1, n - 1):
        if (not mask[k] and mask[k - 1] and mask[k + 1]
                and _latin(tokens[k - 1]) and _latin(tokens[k + 1])
                and any(_is_function_word(tokens[k], lang)
                        for lang in {first_language, second_language, "English"})):
            mask[k] = True

    def joins(tok: str, to_the_right: bool) -> bool:
        if to_the_right and tok.isdigit():
            return True
        return (_latin(tok) and tok[:1].isupper()
                and _word_side(tok, first_language, second_language) != first_language)

    changed = True
    while changed:
        changed = False
        for k in range(n):
            if mask[k]:
                continue
            after_name = k > 0 and mask[k - 1] and (_latin(tokens[k - 1]) or tokens[k - 1].isdigit())
            before_name = k + 1 < n and mask[k + 1] and _latin(tokens[k + 1])
            if (after_name and joins(tokens[k], True)) or (before_name and joins(tokens[k], False)):
                mask[k] = changed = True
    return mask


def _context_language(out, ent, i, j, own, langs):
    """An opaque name (tokens i..j-1) may not open a run of its own.

    Inside one language it takes that language. At a real switch it joins the
    side not cut off by punctuation; if neither or both are, it keeps ``own``
    when that is one of the two sides.
    """
    n = len(out)

    def neighbour(k, step):
        k += step
        cut = False
        while 0 <= k < n:
            t, lang = out[k]
            if not _is_word(t):
                cut = True
            elif not ent[k] and lang in langs:
                return lang, cut
            k += step
        return None, cut

    (prev, prev_cut), (nxt, nxt_cut) = neighbour(i, -1), neighbour(j - 1, 1)
    if prev and nxt and prev != nxt:
        if prev_cut != nxt_cut:
            return nxt if prev_cut else prev
        return own if own in (prev, nxt) else prev
    return prev or nxt or own


def _spans(ent):
    i, n = 0, len(ent)
    while i < n:
        if not ent[i]:
            i += 1
            continue
        j = i
        while j < n and ent[j]:
            j += 1
        yield i, j
        i = j


def resolve_names(tagged: list[tuple[str, str]], is_entity, first_language: str,
                  second_language: str, names: str = DEFAULT_NAMES) -> list[tuple[str, str]]:
    """Apply a names policy to language-tagged tokens (see the module docstring)."""
    if is_entity is None or names == "language":
        return list(tagged)
    if names not in NAME_POLICIES:
        raise ValueError(f"unknown names policy {names!r}; use one of {NAME_POLICIES}")
    toks = [t for t, _ in tagged]
    ent = repair_spans(toks, is_entity, first_language, second_language)
    if names == "transparent":
        return apply_entities(tagged, ent)

    from .dispatch import _hindi_is_devanagari
    from .routing import SCRIPT_DISJOINT_CONFIRMED
    script_row = (first_language in SCRIPT_DISJOINT_CONFIRMED
                  or (first_language == "Hindi" and _hindi_is_devanagari(toks)))
    langs = (first_language, second_language)
    out = list(tagged)
    for i, j in _spans(ent):
        words = [k for k in range(i, j) if _is_word(out[k][0])]
        tagged_langs = [out[k][1] for k in words if out[k][1] in langs]
        if not tagged_langs:
            continue
        own = max(langs, key=tagged_langs.count)
        latin = any(_latin(out[k][0]) for k in words)

        if names == "hybrid":
            title = len(words) >= 2 and any(_is_function_word(out[k][0], own) for k in words)
            if not title:
                own = _context_language(out, ent, i, j, own, langs)
        elif names == "script":
            if latin and second_language == "English":
                own = second_language
        elif names == "script_words" and latin:
            if script_row:
                if second_language == "English":
                    own = second_language
            else:
                votes = [_word_side(out[k][0], first_language, second_language) for k in words]
                votes = [v for v in votes if v]
                if votes and votes.count(langs[0]) != votes.count(langs[1]):
                    own = max(langs, key=votes.count)
                else:
                    own = _context_language(out, ent, i, j, own, langs)
        for k in words:
            out[k] = (out[k][0], own)
    return out

def apply_entities(tagged: list[tuple[str, str]],
                   is_entity: list[bool]) -> list[tuple[str, str]]:
    """The ``transparent`` policy: relabel entity *words* ``NE``.

    Punctuation inside a name stays ``other``.
    """
    return [(t, NE if ent and _is_word(t) else lang)
            for (t, lang), ent in zip(tagged, is_entity)]


def absorb_other(tagged: list[tuple[str, str]]) -> list[tuple[str, str]]:
    """Reassign ``other`` tokens to the surrounding language.

    An ``other`` run takes the language of the token before it, unless that
    would merge two different languages - at a genuine switch boundary the
    punctuation joins the *following* language instead, so the boundary stays
    where it is. Leading ``other`` tokens join whatever follows. Token count is
    preserved.
    """
    out = list(tagged)
    n = len(out)
    i = 0
    while i < n:
        if out[i][1] != "other":
            i += 1
            continue
        j = i
        while j < n and out[j][1] == "other":
            j += 1
        # Look through NE tokens: they belong to neither neighbour, so the
        # boundary test must be made against the nearest real language.
        p = i - 1
        while p >= 0 and out[p][1] == NE:
            p -= 1
        q = j
        while q < n and out[q][1] in ("other", NE):
            q += 1
        prev_lang = out[p][1] if p >= 0 else None
        next_lang = out[q][1] if q < n else None
        if prev_lang is None:
            fill = next_lang
        elif next_lang is None or prev_lang == next_lang:
            fill = prev_lang
        else:
            # Genuine switch boundary: do not let punctuation bridge it.
            fill = next_lang
        if fill is not None:
            for k in range(i, j):
                out[k] = (out[k][0], fill)
        i = j
    return out


def to_segments(tagged_tokens: list[tuple[str, str]]) -> list[list]:
    """Collapse consecutive same-language tokens into ``[language, length]`` runs."""
    segments: list[list] = []
    for _, lang in tagged_tokens:
        if segments and segments[-1][0] == lang:
            segments[-1][1] += 1
        else:
            segments.append([lang, 1])
    return segments


def l2_runs(segments: list[list], second_language: str) -> list[int]:
    """Run lengths straight off the segments, punctuation included.

    Prefer :func:`l2_word_runs` for the switch-length question - segments carry
    absorbed punctuation, which inflates every run.
    """
    return [n for lang, n in segments if lang == second_language]


def _is_word(token: str) -> bool:
    return any(ch.isalpha() for ch in token)


def l2_word_runs(tagged: list[tuple[str, str]], second_language: str) -> list[int]:
    """Lengths of each consecutive second-language run, counting WORDS only.

    This is the original question: how many second-language tokens are used
    consecutively before switching back or ending.

    Punctuation is *transparent* - it neither breaks a run nor counts toward
    it. That matters because ``absorb_other`` hands boundary punctuation to the
    following language, so a sentence-final "." belonging to the first-language
    sentence would otherwise be counted as a second-language token and inflate
    every run by one.
    """
    runs, current = [], 0
    for token, lang in tagged:
        if not _is_word(token) or lang == NE:
            continue                      # transparent: no break, no count
        if lang == second_language:
            current += 1
        elif current:
            runs.append(current)
            current = 0
    if current:
        runs.append(current)
    return runs


def parse_utterances(cell: str) -> list[str]:
    """``data_generation_result`` is a stringified Python list of utterances."""
    try:
        value = ast.literal_eval(str(cell))
    except (ValueError, SyntaxError):
        return [str(cell)]
    if isinstance(value, list):
        return [str(v) for v in value]
    return [str(value)]


def tag_row(cell: str, first_language: str, second_language: str,
            cross_utterance: bool = True, is_entity: list[bool] | None = None,
            return_tagged: bool = False, ane_labels: list[str] | None = None,
            names: str = DEFAULT_NAMES) -> dict:
    """Tokenise, tag and run-length-encode one dataframe row.

    ``cross_utterance=True`` streams every utterance together, so a run may span
    a turn boundary; the utterance each run starts in is still recorded.
    """
    from .dispatch import row_strategy, tag_tokens

    utterances = parse_utterances(cell)

    tokens: list[str] = []
    origin: list[int] = []
    for idx, utt in enumerate(utterances):
        toks = tokenize_mixed(utt)
        tokens.extend(toks)
        origin.extend([idx] * len(toks))

    if not tokens:
        return dict(segments=[], l2_runs=[], n_switches=0, max_run=0,
                    total_l2_tokens=0, total_tokens=0, l2_token_ratio=0.0,
                    method=route(first_language, second_language),
                    unknown_rate=0.0, run_utterance_idx=[])

    strategy = row_strategy(tokens, first_language, second_language)
    tagged = tag_tokens(tokens, first_language, second_language,
                        ane_labels=ane_labels, is_entity=is_entity)
    lang_tagged = tagged                  # stored labels: always a language, policy-free
    tagged = resolve_names(tagged, is_entity, first_language, second_language, names)
    if strategy == "fasttext":
        from .model_tagger import unknown_rate
        unk = unknown_rate(tokens, first_language, second_language)
    else:
        unk = 0.0

    if not cross_utterance:
        # Break runs at turn boundaries by inserting a sentinel language.
        tagged = [
            t if i == 0 or origin[i] == origin[i - 1] else (t[0], t[1])
            for i, t in enumerate(tagged)
        ]

    tagged = absorb_other(tagged)
    segments = to_segments(tagged)
    runs = l2_word_runs(tagged, second_language)

    # Utterance index where each second-language run begins.
    run_start_idx: list[int] = []
    pos = 0
    for lang, n in segments:
        if lang == second_language:
            run_start_idx.append(origin[pos] if pos < len(origin) else -1)
        pos += n

    total_l2 = sum(runs)
    counted = sum(n for lang, n in segments if lang not in ("other", NE))
    result = dict(
        segments=segments,
        l2_runs=runs,
        n_switches=len(runs),
        max_run=max(runs) if runs else 0,
        total_l2_tokens=total_l2,
        total_tokens=len(tokens),
        l2_token_ratio=(total_l2 / counted) if counted else 0.0,
        method=strategy,
        unknown_rate=unk,
        run_utterance_idx=run_start_idx,
    )
    if return_tagged:
        result["tagged"] = lang_tagged   # pre-absorption, never NE
    return result

In [ ]:
%%writefile src/anticipation.py
"""Anticipatory labels: will the NEXT token switch language, and for how long?

The tagging pipeline answers "what language is token t?". This module turns
that into the *prediction* task: standing at token t with only tokens 1..t
visible, predict two things about t+1.

    y_sw[t]  = 1 if lang(t+1) != lang(t)          binary, defined for t < n-1
    y_dur[t] = class of the length of the run t+1 starts (Small 1-2 | Medium 3-6 | Large 7+)

Both labels are derived, never annotated - they are a deterministic function of
the token-language sequence produced by ``src.dispatch``. That means label
quality is exactly tagger quality, which is why ``quality`` travels with every
row (``exact`` for script-disjoint pairs, ``provisional`` otherwise).

Three decisions that materially change the numbers, made explicit:

1. **Punctuation is dropped, not labelled.** ``l2_word_runs`` treats
   non-alphabetic tokens as transparent - they neither break a run nor count
   toward it. The anticipatory stream uses the same rule, so a run length here
   means the same thing as a run length in the descriptive statistics. Feeding
   punctuation to the model as a predictable "switch" would inflate the
   positive class with tokens nobody wants predicted.
2. **The final token carries no label.** There is no t+1 to predict, so
   position n-1 is masked out of the loss rather than being assigned a
   convenient 0. Assigning 0 would add one free true-negative per row -
   234,172 of them - and quietly lift precision.
3. **Durations at the sequence end are right-censored.** A run still in
   progress when the text ends is *at least* its observed length. It is
   flagged ``censored``, and it is used as a duration target only when that
   lower bound already fixes the class (7+ is Large).
"""
from __future__ import annotations

import numpy as np

from .segments import DEFAULT_NAMES, absorb_other, resolve_names

# Language slots. The stream is binarised relative to the row's own pair, so a
# model trained across pairs sees "first vs second language", not "Spanish vs
# English" - that is what makes the universality claim testable.
L1, L2 = 0, 1

# A piece or token that carries no language: punctuation, digits, a bare
# SentencePiece "▁". It stays in the input but is transparent to the labels -
# it neither breaks a run nor counts toward one. Equal to streaming.PAD_LANG.
NONE = 2

# Duration classes from the project requirement: Small 1-2, Medium 3-6,
# Large 7+ tokens. The integer length is kept as y_len; the class is derived.
DUR_EDGES = (2, 6)                     # inclusive upper bounds of small, medium
DUR_LABELS = ["small", "medium", "large"]
N_DUR = len(DUR_LABELS)
LARGE_MIN = DUR_EDGES[-1] + 1          # a cut-off run this long is certainly large

def is_word(token: str) -> bool:
    """A token counts toward a run only if it contains a letter."""
    return any(ch.isalpha() for ch in token)


# How named entities count - see segments.py. Scripts set this from --names.
NAMES = DEFAULT_NAMES


def token_slots(tokens, token_lang, first_language: str, second_language: str,
                is_entity=None, names: str | None = None) -> list[int]:
    """Language slot (L1 / L2 / NONE) for every stored token of a row.

    Applies the names policy and ``absorb_other`` exactly as the word stream
    always has; a token with no letters, or no resolvable language, is NONE.
    """
    tagged = resolve_names(list(zip(tokens, token_lang)), is_entity,
                           first_language, second_language, names or NAMES)
    out = []
    for tok, lang in absorb_other(tagged):
        if not is_word(tok):
            out.append(NONE)
        elif lang == second_language:
            out.append(L2)
        elif lang == first_language:
            out.append(L1)
        else:
            out.append(NONE)                # NE, or unresolved: drop rather than guess
    return out


def word_stream(tokens, token_lang, first_language: str, second_language: str,
                is_entity=None, names: str | None = None):
    """Collapse a tagged row into the (words, language-slot) stream to model.

    Keeps only tokens with a language slot, so punctuation and digits are
    dropped from the stream entirely.
    """
    slots = token_slots(tokens, token_lang, first_language, second_language,
                        is_entity, names)
    keep = [i for i, s in enumerate(slots) if s != NONE]
    return ([tokens[i] for i in keep],
            np.asarray([slots[i] for i in keep], dtype=np.int8))

def run_starts_and_lengths(langs: np.ndarray):
    """Index where each run begins, and its length. Length n == 0 -> empty."""
    n = len(langs)
    if n == 0:
        return np.empty(0, np.int64), np.empty(0, np.int64)
    boundary = np.flatnonzero(langs[1:] != langs[:-1]) + 1
    starts = np.concatenate([[0], boundary])
    ends = np.concatenate([boundary, [n]])
    return starts, ends - starts


def dur_class(length: int) -> int:
    """Map an integer run length to its DUR_LABELS index."""
    for i, upper in enumerate(DUR_EDGES):
        if length <= upper:
            return i
    return N_DUR - 1


def anticipatory_labels(langs: np.ndarray) -> dict:
    """Per-position labels for a language-slot sequence; NONE slots are transparent.

    Position ``t`` describes the next language-bearing position after it.

    * ``y_sw``      0/1 switch indicator (-1 where undefined: NONE slots, last one)
    * ``y_len``     integer length of the run the switch starts (0 elsewhere)
    * ``y_dur``     ``dur_class(y_len)`` (-1 elsewhere)
    * ``censored``  that run is still open at the end of the sequence
    * ``dur_known`` a duration target exists and its class is certain
    """
    n = len(langs)
    y_sw = np.full(n, -1, dtype=np.int8)
    y_len = np.zeros(n, dtype=np.int32)
    y_dur = np.full(n, -1, dtype=np.int8)
    censored = np.zeros(n, dtype=bool)
    dur_known = np.zeros(n, dtype=bool)
    out = dict(y_sw=y_sw, y_len=y_len, y_dur=y_dur, censored=censored,
               dur_known=dur_known)
    idx = np.flatnonzero(np.asarray(langs) != NONE)
    if len(idx) < 2:
        return out
    lb = np.asarray(langs)[idx]
    y_sw[idx[:-1]] = (lb[1:] != lb[:-1]).astype(np.int8)
    starts, lengths = run_starts_and_lengths(lb)
    m = len(lb)
    # A run starting at language-bearing index s>0 is anticipated at idx[s-1].
    for s, ln in zip(starts[1:], lengths[1:]):
        t = idx[s - 1]
        cut = bool(s + ln == m)
        y_len[t] = ln
        y_dur[t] = dur_class(int(ln))
        censored[t] = cut
        dur_known[t] = (not cut) or ln >= LARGE_MIN
    return out

def row_example(row) -> dict | None:
    """Build one word-level training example from a ``load_tagged``-shaped row."""
    words, langs = word_stream(
        row["tokens"], row["token_lang"],
        row["first_language"], row["second_language"],
        is_entity=row.get("is_entity"),
    )
    if len(words) < 2:
        return None
    return dict(
        row_id=int(row["row_id"]),
        words=words,
        langs=langs,
        first_language=row["first_language"],
        second_language=row["second_language"],
        quality=row.get("quality", "unknown"),
        **anticipatory_labels(langs),
    )

In [ ]:
%%writefile src/subword.py
"""Sub-word pieces for the anticipation task.

Each XLM-R piece takes the language of the word it sits in. The word labels
are the final per-token labels (``token_lang`` after the names policy), and
the original text comes from the SwitchLingua CSVs, joined on ``row_id``. The
stored tokens were produced by ``tokenize_mixed`` from that text, so an
in-order search recovers each token's character span exactly.

A piece's language is the majority slot of the word characters it covers
(ties go to L1; a piece covering two languages is counted, not an error). A
piece covering no word characters - punctuation, digits, spaces, or the bare
SentencePiece "▁" that XLM-R emits with the next character's offsets - is
NONE and transparent to the labels. Merged CJK pieces such as 東京 need no
special case: their characters share a language.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from .anticipation import L1, L2, NONE, anticipatory_labels, token_slots
from .tokenizer import tokenize_mixed

XLMR_VOCAB = 250_002
LABEL_KEYS = ("y_sw", "y_len", "y_dur", "censored", "dur_known")


def align_words(utterances: list[str], tokens: list[str]):
    """``(utterance, start, end)`` for every stored token, in order, or None."""
    spans, k = [], 0
    for u, text in enumerate(utterances):
        pos = 0
        for t in tokenize_mixed(text):
            if k >= len(tokens) or tokens[k] != t:
                return None
            j = text.find(t, pos)
            if j < 0:
                return None
            spans.append((u, j, j + len(t)))
            pos = j + len(t)
            k += 1
    return spans if k == len(tokens) else None


def piece_languages(offsets, blank, char_slots: np.ndarray):
    """Slot per piece from the word slots of the characters it covers."""
    langs, mixed = [], 0
    for (s, e), is_blank in zip(offsets, blank):
        if is_blank:
            langs.append(NONE)
            continue
        seg = char_slots[s:e]
        n1, n2 = int((seg == L1).sum()), int((seg == L2).sum())
        if n1 == 0 and n2 == 0:
            langs.append(NONE)
            continue
        mixed += int(n1 > 0 and n2 > 0)
        langs.append(L2 if n2 > n1 else L1)
    return langs, mixed


def subword_row(utterances, tokens, token_lang, first_language: str,
                second_language: str, tokenizer, is_entity=None):
    """Piece ids and per-piece language slots for one row, or None if unaligned."""
    tokens, token_lang = list(tokens), list(token_lang)
    spans = align_words(utterances, tokens)
    if spans is None:
        return None
    slots = token_slots(tokens, token_lang, first_language, second_language,
                        None if is_entity is None else list(is_entity))
    char_slots = [np.full(len(u), -1, dtype=np.int8) for u in utterances]
    for (u, s, e), slot in zip(spans, slots):
        if slot != NONE:
            char_slots[u][s:e] = slot
    ids, pieces, langs, mixed = [], [], [], 0
    for u, text in enumerate(utterances):
        if not text:
            continue
        enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
        strs = tokenizer.convert_ids_to_tokens(enc["input_ids"])
        blank = [p.replace("▁", "") == "" for p in strs]
        lg, m = piece_languages(enc["offset_mapping"], blank, char_slots[u])
        ids.extend(enc["input_ids"])
        pieces.extend(strs)
        langs.extend(lg)
        mixed += m
    return dict(piece_ids=ids, pieces=pieces, piece_lang=langs, mixed_pieces=mixed)


def build_frame(frame: pd.DataFrame, texts: dict, tokenizer, max_fail: float = 0.001):
    """Sub-word rows for every tagged row; misaligned rows are skipped and counted."""
    out, failed, mixed = [], 0, 0
    has_ent = "is_entity" in frame.columns
    for r in frame.itertuples(index=False):
        res = subword_row(texts[int(r.row_id)], r.tokens, r.token_lang,
                          r.first_language, r.second_language, tokenizer,
                          is_entity=r.is_entity if has_ent else None)
        if res is None:
            failed += 1
            continue
        mixed += res["mixed_pieces"]
        lab = anticipatory_labels(np.asarray(res["piece_lang"], dtype=np.int8))
        out.append(dict(row_id=int(r.row_id), first_language=r.first_language,
                        second_language=r.second_language, quality=r.quality, **res,
                        **{k: lab[k].tolist() for k in LABEL_KEYS}))
    rate = failed / max(len(frame), 1)
    if rate > max_fail:
        raise RuntimeError(f"{failed:,} of {len(frame):,} rows failed to align "
                           f"({rate:.2%} > {max_fail:.2%})")
    stats = dict(rows=len(out), failed=failed, mixed_pieces=mixed,
                 pieces=sum(len(x["piece_ids"]) for x in out))
    return pd.DataFrame(out), stats


def write_parquet(frame: pd.DataFrame, path: str, row_group_size: int = 8192) -> None:
    import pyarrow as pa
    import pyarrow.parquet as pq
    schema = pa.schema([
        ("row_id", pa.int64()), ("first_language", pa.string()),
        ("second_language", pa.string()), ("quality", pa.string()),
        ("pieces", pa.list_(pa.string())), ("piece_ids", pa.list_(pa.int32())),
        ("piece_lang", pa.list_(pa.int8())),
        ("y_sw", pa.list_(pa.int8())), ("y_len", pa.list_(pa.int32())),
        ("y_dur", pa.list_(pa.int8())), ("censored", pa.list_(pa.bool_())),
        ("dur_known", pa.list_(pa.bool_())),
        ("mixed_pieces", pa.int32()),
    ])
    table = pa.Table.from_pandas(frame[schema.names], schema=schema, preserve_index=False)
    pq.write_table(table, path, compression="snappy", row_group_size=row_group_size)

## Download the word labels

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

path = hf_hub_download(LABELS_REPO, "tagged.parquet", repo_type="dataset", token=TOKEN)
cols = ["row_id", "first_language", "second_language", "quality", "tokens", "token_lang", "is_entity"]
df = pd.read_parquet(path, columns=cols)
print(f"downloaded {len(df):,} rows")

split_of = lambda rid: "train" if rid % 10 < 8 else "val" if rid % 10 == 8 else "test"
df["split"] = df.row_id.map(split_of)
if PAIRS:
    keep = {tuple(p) for p in PAIRS}
    df = df[[(a, b) in keep for a, b in zip(df.first_language, df.second_language)]]
if SPLITS:
    df = df[df.split.isin(SPLITS)]
if LIMIT:
    df = df.groupby(["first_language", "second_language"], sort=False).head(LIMIT)
df = df.reset_index(drop=True)
print(f"selected {len(df):,} rows")
print(df.groupby(["first_language", "second_language"]).size().to_string())

## Build the labels you chose

In [ ]:
import numpy as np
import src.anticipation as A
from src.anticipation import anticipatory_labels, token_slots

A.NAMES = NAMES                                   # the names policy every label call uses
LANG = np.array(["L1", "L2", "-"])
CLASS = {-1: "", 0: "small", 1: "medium", 2: "large"}
KEYS = ["y_sw", "y_len", "y_dur", "censored", "dur_known"]

def readable(unit_col, units, slots):
    lab = anticipatory_labels(np.asarray(slots, dtype=np.int8))
    return {unit_col: list(units), "lang": LANG[np.asarray(slots)].tolist(),
            "y_sw": lab["y_sw"].tolist(), "y_len": lab["y_len"].tolist(),
            "y_dur": [CLASS[int(d)] for d in lab["y_dur"]],
            "censored": lab["censored"].tolist(), "dur_known": lab["dur_known"].tolist()}

meta = ["row_id", "first_language", "second_language", "split", "quality"]
rows = []
if LEVEL == "word":
    UNIT = "tokens"
    for r in df.itertuples(index=False):
        slots = token_slots(list(r.tokens), list(r.token_lang), r.first_language,
                            r.second_language, list(r.is_entity))
        rows.append({**{k: getattr(r, k) for k in meta}, **readable(UNIT, r.tokens, slots),
                     "is_entity": [int(e) for e in r.is_entity]})
else:
    UNIT = "pieces"
    import glob
    from huggingface_hub import snapshot_download
    from transformers import AutoTokenizer
    from src.segments import parse_utterances
    from src.subword import build_frame

    # The original text: same file order as the local pipeline, so row_id matches.
    snap = snapshot_download("Shelton1013/SwitchLingua_text", repo_type="dataset",
                             allow_patterns=["*.csv"], token=TOKEN)
    files = sorted(glob.glob(os.path.join(snap, "*.csv")), key=os.path.basename)
    corpus = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    assert len(corpus) == 234_172, f"unexpected corpus size {len(corpus):,}"
    texts = {int(i): parse_utterances(corpus.data_generation_result.iloc[int(i)]) for i in df.row_id}
    tok = AutoTokenizer.from_pretrained("xlm-roberta-base")
    sw, stats = build_frame(df, texts, tok)
    print(f"{stats['rows']:,} rows, {stats['pieces']:,} pieces, "
          f"{stats['failed']} alignment failures, {stats['mixed_pieces']} mixed pieces")
    split = dict(zip(df.row_id, df.split))
    for r in sw.itertuples(index=False):
        rows.append({"row_id": r.row_id, "first_language": r.first_language,
                     "second_language": r.second_language, "split": split[r.row_id],
                     "quality": r.quality, **readable(UNIT, r.pieces, r.piece_lang),
                     "piece_ids": list(r.piece_ids)})
labels = pd.DataFrame(rows)
print(f"labelled {len(labels):,} rows at the {LEVEL} level, names policy {NAMES!r}")

## Per-pair summary

In [ ]:
def pair_summary(g):
    sw = np.concatenate([np.asarray(x) for x in g.y_sw])
    known = np.concatenate([np.asarray(x) for x in g.dur_known])
    cls = np.concatenate([np.asarray(x, dtype=object) for x in g.y_dur])[known]
    n = max(len(cls), 1)
    return pd.Series({"rows": len(g), "scored": int((sw >= 0).sum()),
                      "switch rate": f"{(sw == 1).sum() / max((sw >= 0).sum(), 1):.1%}",
                      "small": f"{(cls == 'small').sum() / n:.0%}",
                      "medium": f"{(cls == 'medium').sum() / n:.0%}",
                      "large": f"{(cls == 'large').sum() / n:.0%}"})

print(labels.groupby(["first_language", "second_language"]).apply(pair_summary).to_string())

## Look at a row

In [ ]:
# One row, token by token. Change `i` to look at others.
i = 0
r = labels.iloc[i]
print(f"row {r.row_id} | {r.first_language} -> {r.second_language} | {r.split}")
pd.DataFrame({k: r[k] for k in [UNIT, "lang", "y_sw", "y_len", "y_dur", "censored"]}).head(40)

## Save

In [ ]:
name = f"switchlingua_labels_{LEVEL}_{NAMES}"
out = []
if EXPORT in ("parquet", "both"):
    labels.to_parquet(os.path.join(WORK, name + ".parquet"), index=False)
    out.append(name + ".parquet")
if EXPORT in ("csv", "both"):
    # One line per conversation; each list column is space-separated, token-aligned.
    flat = labels.copy()
    for c in [UNIT, "lang", "y_sw", "y_len", "y_dur", "censored", "dur_known"]:
        flat[c] = [" ".join(str(int(v)) if isinstance(v, (bool, np.bool_)) else (str(v) or "_") for v in xs)
                   for xs in flat[c]]
    flat = flat.drop(columns=[c for c in ("piece_ids", "is_entity") if c in flat])
    flat.to_csv(os.path.join(WORK, name + ".csv"), index=False)
    out.append(name + ".csv")
for f in out:
    print(f"wrote {os.path.join(WORK, f)}  ({os.path.getsize(os.path.join(WORK, f)) / 1e6:.0f} MB)")
print("In the CSV, an empty y_dur is written as '_' so the columns stay aligned.")